In [1]:

# ============================================================
# PRACTICAL 1: IMPROVE A BASELINE CLASSIFIER
# Dataset: Breast Cancer Dataset
# ============================================================


# TASK 0: IMPORT REQUIRED LIBRARIES

import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# TASK 1: LOAD DATASET AND CREATE TRAIN, VALIDATION AND TEST SPLITS
# ============================================================

# Load the breast cancer dataset

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Dataset loaded successfully!")
print("Dataset Shape:", X.shape)

print("\nTarget Classes:")
print(data.target_names)

# Split data into training (60%) and temporary (40%)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

# Split temporary data into validation (20%) and test (20%)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nData Split Successfully!")

print("Training Data:", X_train.shape)
print("Validation Data:", X_val.shape)
print("Testing Data:", X_test.shape)


# ============================================================
# TASK 2: TRAIN LOGISTIC REGRESSION AS BASELINE
# ============================================================

# Create baseline Logistic Regression model

baseline_model = LogisticRegression(max_iter=10000)

# Train the model

baseline_model.fit(X_train, y_train)

# Predict validation data

baseline_predictions = baseline_model.predict(X_val)

# Calculate accuracy

baseline_accuracy = accuracy_score(
    y_val,
    baseline_predictions
)

print("\nBASELINE LOGISTIC REGRESSION")
print("Baseline Accuracy:", round(baseline_accuracy, 4))


# ============================================================
# TASK 3: USE 5-FOLD CROSS VALIDATION
# ============================================================

# Create a pipeline for cross-validation

cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=10000))
])

# Perform 5-fold cross-validation on training data

cv_scores = cross_val_score(
    cv_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("\n5-FOLD CROSS VALIDATION RESULTS")

for i, score in enumerate(cv_scores, start=1):
    print(f"Fold {i} Accuracy: {score:.4f}")

print("Mean Cross Validation Accuracy:",
      round(cv_scores.mean(), 4))


# ============================================================
# TASK 4: CREATE PIPELINE WITH STANDARD SCALER
#         AND LOGISTIC REGRESSION
# ============================================================

# Create machine learning pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=10000))
])

# Train the pipeline

pipeline.fit(X_train, y_train)

# Predict validation data

pipeline_predictions = pipeline.predict(X_val)

# Calculate pipeline accuracy

pipeline_accuracy = accuracy_score(
    y_val,
    pipeline_predictions
)

print("\nSTANDARD SCALER + LOGISTIC REGRESSION PIPELINE")

print("Pipeline Accuracy:",
      round(pipeline_accuracy, 4))


# ============================================================
# TASK 5: HYPERPARAMETER TUNING USING GRIDSEARCHCV
# ============================================================

# Define hyperparameter values

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

# Create GridSearchCV

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train GridSearchCV using training data

grid_search.fit(X_train, y_train)

# Get best parameters

print("\nGRID SEARCH RESULTS")

print("Best Parameters:", grid_search.best_params_)

print("Best Cross Validation Score:",
      round(grid_search.best_score_, 4))

# Predict validation data using the best model

best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_val)

tuned_accuracy = accuracy_score(
    y_val,
    tuned_predictions
)

print("Tuned Validation Accuracy:",
      round(tuned_accuracy, 4))


# ============================================================
# FINAL COMPARISON TABLE
# ============================================================

results = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "5-Fold Cross Validation Mean",
        "StandardScaler + Logistic Regression",
        "GridSearchCV Tuned Model"
    ],

    "Accuracy": [
        baseline_accuracy,
        cv_scores.mean(),
        pipeline_accuracy,
        tuned_accuracy
    ]
})

print("\nFINAL MODEL COMPARISON")

print(results.to_string(index=False))


# ============================================================
# FINAL TESTING
# Train the best model on training + validation data
# and evaluate once on the unseen test data.
# ============================================================

X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

final_model = grid_search.best_estimator_

final_model.fit(X_train_final, y_train_final)

test_predictions = final_model.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print("\nFINAL TEST RESULTS")

print("Test Accuracy:", round(test_accuracy, 4))

print("\nClassification Report:")

print(classification_report(
    y_test,
    test_predictions,
    target_names=data.target_names
))


# ============================================================
# CONCLUSION
# ============================================================

print("\nCONCLUSION")

if tuned_accuracy > baseline_accuracy:
    print("Hyperparameter tuning improved validation accuracy.")
elif tuned_accuracy == baseline_accuracy:
    print("Both models achieved the same validation accuracy.")
else:
    print("Baseline model achieved better validation accuracy.")

print("The best model was evaluated using the unseen test dataset.")


Dataset loaded successfully!
Dataset Shape: (569, 30)

Target Classes:
['malignant' 'benign']

Data Split Successfully!
Training Data: (341, 30)
Validation Data: (114, 30)
Testing Data: (114, 30)

BASELINE LOGISTIC REGRESSION
Baseline Accuracy: 0.9649

5-FOLD CROSS VALIDATION RESULTS
Fold 1 Accuracy: 0.9565
Fold 2 Accuracy: 0.9706
Fold 3 Accuracy: 0.9559
Fold 4 Accuracy: 0.9706
Fold 5 Accuracy: 0.9853
Mean Cross Validation Accuracy: 0.9678

STANDARD SCALER + LOGISTIC REGRESSION PIPELINE
Pipeline Accuracy: 0.9912

GRID SEARCH RESULTS
Best Parameters: {'classifier__C': 0.1}
Best Cross Validation Score: 0.9736
Tuned Validation Accuracy: 0.9825

FINAL MODEL COMPARISON
                               Model  Accuracy
        Baseline Logistic Regression  0.964912
        5-Fold Cross Validation Mean  0.967775
StandardScaler + Logistic Regression  0.991228
            GridSearchCV Tuned Model  0.982456

FINAL TEST RESULTS
Test Accuracy: 0.9737

Classification Report:
              precision   

In [1]:

# ============================================================
# PRACTICAL 1: IMPROVE A BASELINE CLASSIFIER
# Dataset: Breast Cancer Dataset
# ============================================================


# TASK 0: IMPORT REQUIRED LIBRARIES

import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# TASK 1: LOAD DATASET AND CREATE TRAIN, VALIDATION AND TEST SPLITS
# ============================================================

# Load the breast cancer dataset

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Dataset loaded successfully!")
print("Dataset Shape:", X.shape)

print("\nTarget Classes:")
print(data.target_names)

# Split data into training (60%) and temporary (40%)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

# Split temporary data into validation (20%) and test (20%)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nData Split Successfully!")

print("Training Data:", X_train.shape)
print("Validation Data:", X_val.shape)
print("Testing Data:", X_test.shape)


# ============================================================
# TASK 2: TRAIN LOGISTIC REGRESSION AS BASELINE
# ============================================================

# Create baseline Logistic Regression model

baseline_model = LogisticRegression(max_iter=10000)

# Train the model

baseline_model.fit(X_train, y_train)

# Predict validation data

baseline_predictions = baseline_model.predict(X_val)

# Calculate accuracy

baseline_accuracy = accuracy_score(
    y_val,
    baseline_predictions
)

print("\nBASELINE LOGISTIC REGRESSION")
print("Baseline Accuracy:", round(baseline_accuracy, 4))


# ============================================================
# TASK 3: USE 5-FOLD CROSS VALIDATION
# ============================================================

# Create a pipeline for cross-validation

cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=10000))
])

# Perform 5-fold cross-validation on training data

cv_scores = cross_val_score(
    cv_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("\n5-FOLD CROSS VALIDATION RESULTS")

for i, score in enumerate(cv_scores, start=1):
    print(f"Fold {i} Accuracy: {score:.4f}")

print("Mean Cross Validation Accuracy:",
      round(cv_scores.mean(), 4))


# ============================================================
# TASK 4: CREATE PIPELINE WITH STANDARD SCALER
#         AND LOGISTIC REGRESSION
# ============================================================

# Create machine learning pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=10000))
])

# Train the pipeline

pipeline.fit(X_train, y_train)

# Predict validation data

pipeline_predictions = pipeline.predict(X_val)

# Calculate pipeline accuracy

pipeline_accuracy = accuracy_score(
    y_val,
    pipeline_predictions
)

print("\nSTANDARD SCALER + LOGISTIC REGRESSION PIPELINE")

print("Pipeline Accuracy:",
      round(pipeline_accuracy, 4))


# ============================================================
# TASK 5: HYPERPARAMETER TUNING USING GRIDSEARCHCV
# ============================================================

# Define hyperparameter values

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

# Create GridSearchCV

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train GridSearchCV using training data

grid_search.fit(X_train, y_train)

# Get best parameters

print("\nGRID SEARCH RESULTS")

print("Best Parameters:", grid_search.best_params_)

print("Best Cross Validation Score:",
      round(grid_search.best_score_, 4))

# Predict validation data using the best model

best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_val)

tuned_accuracy = accuracy_score(
    y_val,
    tuned_predictions
)

print("Tuned Validation Accuracy:",
      round(tuned_accuracy, 4))


# ============================================================
# FINAL COMPARISON TABLE
# ============================================================

results = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "5-Fold Cross Validation Mean",
        "StandardScaler + Logistic Regression",
        "GridSearchCV Tuned Model"
    ],

    "Accuracy": [
        baseline_accuracy,
        cv_scores.mean(),
        pipeline_accuracy,
        tuned_accuracy
    ]
})

print("\nFINAL MODEL COMPARISON")

print(results.to_string(index=False))


# ============================================================
# FINAL TESTING
# Train the best model on training + validation data
# and evaluate once on the unseen test data.
# ============================================================

X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

final_model = grid_search.best_estimator_

final_model.fit(X_train_final, y_train_final)

test_predictions = final_model.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print("\nFINAL TEST RESULTS")

print("Test Accuracy:", round(test_accuracy, 4))

print("\nClassification Report:")

print(classification_report(
    y_test,
    test_predictions,
    target_names=data.target_names
))


# ============================================================
# CONCLUSION
# ============================================================

print("\nCONCLUSION")

if tuned_accuracy > baseline_accuracy:
    print("Hyperparameter tuning improved validation accuracy.")
elif tuned_accuracy == baseline_accuracy:
    print("Both models achieved the same validation accuracy.")
else:
    print("Baseline model achieved better validation accuracy.")

print("The best model was evaluated using the unseen test dataset.")


Dataset loaded successfully!
Dataset Shape: (569, 30)

Target Classes:
['malignant' 'benign']

Data Split Successfully!
Training Data: (341, 30)
Validation Data: (114, 30)
Testing Data: (114, 30)

BASELINE LOGISTIC REGRESSION
Baseline Accuracy: 0.9649

5-FOLD CROSS VALIDATION RESULTS
Fold 1 Accuracy: 0.9565
Fold 2 Accuracy: 0.9706
Fold 3 Accuracy: 0.9559
Fold 4 Accuracy: 0.9706
Fold 5 Accuracy: 0.9853
Mean Cross Validation Accuracy: 0.9678

STANDARD SCALER + LOGISTIC REGRESSION PIPELINE
Pipeline Accuracy: 0.9912

GRID SEARCH RESULTS
Best Parameters: {'classifier__C': 0.1}
Best Cross Validation Score: 0.9736
Tuned Validation Accuracy: 0.9825

FINAL MODEL COMPARISON
                               Model  Accuracy
        Baseline Logistic Regression  0.964912
        5-Fold Cross Validation Mean  0.967775
StandardScaler + Logistic Regression  0.991228
            GridSearchCV Tuned Model  0.982456

FINAL TEST RESULTS
Test Accuracy: 0.9737

Classification Report:
              precision   